# career_success_score — v34 AutoGluon-only last-chance safe notebook

Son submit hakkı için daha güvenli AutoGluon-only sürüm. Manuel CatBoost/LightGBM/XGBoost/Ridge stack yok; tek öğrenici AutoGluon TabularPredictor.

v33'e göre risk azaltmaları: AutoGluon L2 stack kapalı, ekstra v33 group-percentile rank featureları default kapalı, LB anchor/postprocess yok. Domain featurelar, Türkçe transformer metin sinyali ve test-drift sample weight kalır.


## Plan notu

`$speckit-plan` çağrısı yapıldı fakat bu repo içinde `.speckit/commands/speckit.plan.md` ve `.specify/feature.json` bulunmadı; Spec Kit plan kurulu değil. Bu nedenle pratik çıktı olarak doğrudan final Kaggle notebooku oluşturuldu.


In [ ]:
# Hücre 0 — Kaggle ortam kontrolü ve AutoGluon/NLP paket kurulumu
import importlib.util, os, subprocess, sys

IN_KAGGLE = os.path.exists('/kaggle/working')
FORCE_KAGGLE_PIN_INSTALL = False  # True yaparsan Kaggle'da paketleri pinli sürümlere zorla yeniden kurar.
FIX_P100_TORCH_CU128 = True  # Kaggle P100 + torch cu128 uyumsuzsa cu126 torch kurup kernel restart ister.
PIP_PACKAGES = {
    'autogluon.tabular': 'autogluon.tabular==1.5.0',
    'catboost': 'catboost==1.2.10',  # AutoGluon iç modeli için
    'lightgbm': 'lightgbm==4.6.0',  # AutoGluon iç modeli için
    'xgboost': 'xgboost==3.1.3',    # AutoGluon iç modeli için
    'transformers': 'transformers==4.57.6',
    'sentence_transformers': 'sentence-transformers==5.5.1',
    'accelerate': 'accelerate==1.14.0',
}

print('Python:', sys.version.split()[0], '| executable:', sys.executable)
if IN_KAGGLE:
    print('Kaggle modu: GPU ve Internet açık olmalı. Torch/CUDA yeniden kurulmayacak.')
else:
    print('Local mod: bu dosya Kaggle için hazırlandı ama localde de çalışabilir.')

def module_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

if FORCE_KAGGLE_PIN_INSTALL and IN_KAGGLE:
    install_packages = list(PIP_PACKAGES.values())
else:
    install_packages = [pkg for module, pkg in PIP_PACKAGES.items() if not module_available(module)]

if install_packages:
    print('Kurulacak paketler:', install_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', *install_packages])
    importlib.invalidate_caches()
else:
    print('Gerekli paketler zaten import edilebilir durumda.')

def maybe_fix_p100_torch():
    if not (IN_KAGGLE and FIX_P100_TORCH_CU128):
        return
    try:
        import torch
        if not torch.cuda.is_available():
            return
        gpu_name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| cuda_available:', torch.cuda.is_available())
        print('GPU:', gpu_name, '| capability:', capability)
        needs_fix = capability[0] < 7 and (torch.version.cuda == '12.8' or '+cu128' in torch.__version__)
    except Exception as e:
        print('Torch kontrolü başarısız:', repr(e))
        return
    if not needs_fix:
        return
    print('P100/sm_60 + CUDA 12.8 torch uyumsuz. torch 2.9.1+cu126 kurulacak ve kernel yeniden başlayacak.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '--force-reinstall',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
        'torch==2.9.1+cu126', 'torchvision==0.24.1+cu126'
    ])
    print('Torch değişti. Kaggle kernel yeniden başlatılıyor; sonra Run All tekrar çalıştırın.')
    os.kill(os.getpid(), 9)

maybe_fix_p100_torch()

try:
    import torch
    print('Torch final:', torch.__version__, '| CUDA:', torch.version.cuda, '| cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU final:', torch.cuda.get_device_name(0), '| capability:', torch.cuda.get_device_capability(0))
except Exception as e:
    print('Torch final kontrolü başarısız:', repr(e))


In [ ]:
# Hücre 1 — veri yükleme ve yarışma sözleşmesi
import warnings, time, re, gc, os
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
t0 = time.time()

RANDOM_STATE = 42
NOTEBOOK_VERSION = 'v39_text_xlmr_year_ensemble'
TARGET = 'career_success_score'
ID_COL = 'student_id'
LOCAL_RUN = not Path('/kaggle/working').exists()
OVERNIGHT_GPU_RUN = True  # Kaggle GPU koşusu: transformer OOF açık, AutoGluon daha uzun çalışır.
LOCAL_TIME_LIMIT = 7200  # Local kalite koşusu: 2 saat AutoGluon.
KAGGLE_TIME_LIMIT = 21600 if OVERNIGHT_GPU_RUN else 10800  # Son hak: 3-6 saat, disk/time riski kontrollü.
TIME_LIMIT = LOCAL_TIME_LIMIT if LOCAL_RUN else KAGGLE_TIME_LIMIT

# Feature anahtarları: süreye göre kapatıp açılabilir.
RUN_SENTENCE_EMBEDDINGS = True
SENTENCE_EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SENTENCE_EMBEDDING_COMPONENTS = 128
RUN_TRANSFORMER_TEXT_OOF = bool(OVERNIGHT_GPU_RUN or not LOCAL_RUN)
TRANSFORMER_MODEL_IDS = [
    'dbmdz/bert-base-turkish-cased',
    'dbmdz/electra-base-turkish-cased-discriminator',
    'xlm-roberta-base',
]
TRANSFORMER_N_SPLITS = 5
TRANSFORMER_EPOCHS = 3 if OVERNIGHT_GPU_RUN else 2
TRANSFORMER_MAX_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 12  # XLM-R ekli olduğu için OOM riskini azaltan batch.
TRANSFORMER_LR = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
ALLOW_CPU_TRANSFORMER_OOF = False  # GPU yoksa text fine-tune çok yavaş; sentence embedding + keyword featureları devam eder.
REQUIRE_CUDA_FOR_TEXT_OOF = True  # Gece koşusunda CUDA görünmüyorsa sessiz skip yerine hata ver.
RUN_MANUAL_XGBOOST = False  # Niş karar: manual XGBoost önceki OOF'larda zayıf/yavaş; AutoGluon içindeki XGB ayrı kalır.

CACHE_DIR = Path('/kaggle/working/text_feature_cache') if Path('/kaggle/working').exists() else Path('derived/text_feature_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_READ_DIRS = [CACHE_DIR]
if Path('/kaggle/input').exists():
    for p in Path('/kaggle/input').rglob('*.npz'):
        if p.parent not in CACHE_READ_DIRS:
            CACHE_READ_DIRS.append(p.parent)

def find_cache_file(filename):
    for d in CACHE_READ_DIRS:
        p = d / filename
        if p.exists():
            return p
    return None
try:
    import torch
    HAS_CUDA = bool(torch.cuda.is_available())
    CUDA_DEVICE_COUNT = torch.cuda.device_count()
    CUDA_DEVICE_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else None
except Exception as e:
    HAS_CUDA = False
    CUDA_DEVICE_COUNT = 0
    CUDA_DEVICE_NAME = None
    print('CUDA kontrolü yapılamadı:', repr(e))

CATBOOST_TASK_TYPE = 'GPU' if HAS_CUDA else 'CPU'
CATBOOST_DEVICES = '0'
LIGHTGBM_USE_GPU = False  # Sadece LightGBM GPU destekli kurulduysa True yap; çoğu pip wheel CPU-only gelir.
XGBOOST_USE_GPU = bool(HAS_CUDA)
AG_NUM_BAG_FOLDS = 5
AG_NUM_STACK_LEVELS = 0  # Son hak için L2 stack kapalı: daha stabil, daha az disk/time/overfit riski.
AG_PRESETS = 'best_quality'
AG_EXCLUDED_MODEL_TYPES = ['KNN']  # KNN bütçe israfı; RF/XT dahil, ensemble gerekirse yok sayar.
AGGRESSIVE_LB_MODE = True  # Validasyon en iyi aday açık ara değilse bile ayrıca submission olarak yazılır.
RUN_RESIDUAL_LGB_CALIBRATION = False  # Residual düzeltme default kapalı; candidate olarak denenir ama ana seçimi bozmaz.
REQUIRE_LB827_ANCHOR = False  # Kaliteli notebook bağımsız çalışmalı; anchor input zorunlu değil.
RUN_LB_ANCHOR_EXPORTS = False  # LB-postprocess dosya üretimini default kapat; gerekirse True yap.
AG_WEIGHT_EVALUATION = False  # Resmi MSE'ye yakın seçim için AutoGluon validation unweighted kalsın.
AG_TABULAR_NUM_GPUS = 0  # Tabular GBM/CatBoost GPU kalite uyarısı nedeniyle kapalı; GPU NLP'de kullanılır.
RUN_V33_GROUP_PERCENTILE_FEATURES = False  # Son hak için ekstra transductive rank featureları kapalı; domain v33 featurelar kalır.
RUN_FREQUENCY_ENCODING = False  # Scout: *_freq kolonları public-gap riskini artırıyor; default kapalı.
DROP_CATEGORICAL_INTERACTION_FEATURES = True  # Scout: __ içeren combo featurelar public-shift holdout'ta en net gürültü.
DROP_TOPIC_ROLE_TEXT_FLAGS = True  # Beam scout: mentor_topic_* / mentor_mentions_target_role küçük ama stabil gürültü.
SAMPLE_WEIGHT_GROUPS = ['application_year', 'target_role', 'university_tier']
SAMPLE_WEIGHT_SHRINK = 200.0  # Train/test grup oranlarını 1.0'a doğru shrink eder; aşırı rare-role ağırlığını bastırır.

print(f'Notebook version: {NOTEBOOK_VERSION}')
print(f'LOCAL_RUN={LOCAL_RUN} | OVERNIGHT_GPU_RUN={OVERNIGHT_GPU_RUN} | TIME_LIMIT={TIME_LIMIT} | RUN_TRANSFORMER_TEXT_OOF={RUN_TRANSFORMER_TEXT_OOF}')
print(f'CUDA_VISIBLE={HAS_CUDA} | cuda_device_count={CUDA_DEVICE_COUNT} | device0={CUDA_DEVICE_NAME}')
print(f'CatBoost task={CATBOOST_TASK_TYPE} | LightGBM GPU={LIGHTGBM_USE_GPU} | XGBoost GPU={XGBOOST_USE_GPU}')
print(f'AutoGluon presets={AG_PRESETS} | bag_folds={AG_NUM_BAG_FOLDS} | stack_levels={AG_NUM_STACK_LEVELS} | excluded={AG_EXCLUDED_MODEL_TYPES} | weight_eval={AG_WEIGHT_EVALUATION} | ag_gpus={AG_TABULAR_NUM_GPUS} | extra_rank={RUN_V33_GROUP_PERCENTILE_FEATURES} | freq_encoding={RUN_FREQUENCY_ENCODING} | drop_combo={DROP_CATEGORICAL_INTERACTION_FEATURES} | drop_topic={DROP_TOPIC_ROLE_TEXT_FLAGS}')
print(f'Sample weight groups={SAMPLE_WEIGHT_GROUPS} | shrink={SAMPLE_WEIGHT_SHRINK}')
print('Text cache read dirs:', [str(p) for p in CACHE_READ_DIRS[:8]])


def find_csv(cands, required=True):
    roots = [Path('/kaggle/input'), Path('.'), Path('datathon-2026 (1)')]
    seen = set()
    for root in roots:
        if not root.exists() or root in seen:
            continue
        seen.add(root)
        for c in cands:
            m = sorted(root.rglob(c)) if root.is_dir() else []
            if m:
                return m[0]
    if required:
        raise FileNotFoundError(cands)
    return None


def read_csv(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    return df

train_final_path = find_csv(['train_final.csv'], required=False)
test_final_path = find_csv(['test_final.csv'], required=False)
raw_train_path = find_csv(['train.csv'], required=False)
raw_test_path = find_csv(['test_x.csv', 'test.csv'], required=False)

if train_final_path and test_final_path:
    train = read_csv(train_final_path)
    test = read_csv(test_final_path)
    print('Hazır FE dosyaları kullanılıyor:', train_final_path, test_final_path)
else:
    train = read_csv(raw_train_path)
    test = read_csv(raw_test_path)
    print('Ham yarışma dosyaları kullanılıyor:', raw_train_path, raw_test_path)

raw_train = read_csv(raw_train_path) if raw_train_path else train.copy()
raw_test = read_csv(raw_test_path) if raw_test_path else test.copy()

# Hazır FE dosyaları kullanılsa bile ham veri kolonları lazımsa geri ekle.
# Böylece mentor text / domain raw skorları kaybolmaz.
def enrich_missing_raw_columns(df, raw_df):
    if raw_df is None:
        return df
    missing_cols = [c for c in raw_df.columns if c not in df.columns and c != TARGET]
    if not missing_cols:
        return df
    df = df.copy()
    if len(df) == len(raw_df):
        for c in missing_cols:
            df[c] = raw_df[c].values
        print(f'Eksik raw kolonlar pozisyonel eklendi: {len(missing_cols)}')
        return df
    if ID_COL in df.columns and ID_COL in raw_df.columns:
        add = raw_df[[ID_COL] + missing_cols].drop_duplicates(ID_COL)
        df = df.merge(add, on=ID_COL, how='left')
        print(f'Eksik raw kolonlar ID merge ile eklendi: {len(missing_cols)}')
    return df

train = enrich_missing_raw_columns(train, raw_train)
test = enrich_missing_raw_columns(test, raw_test)

test_ids = raw_test[ID_COL].values if ID_COL in raw_test.columns else test[ID_COL].values
assert TARGET in train.columns, f'{TARGET} train içinde yok'
assert len(test_ids) == len(test), 'test satır sayısı orijinal test ile uyuşmuyor!'

for col in ['age', 'coding_score']:
    if col in raw_test.columns and col in test.columns:
        assert np.allclose(pd.to_numeric(raw_test[col], errors='coerce').fillna(-1).values,
                           pd.to_numeric(test[col], errors='coerce').fillna(-1).values), f'HIZA BOZUK: {col}'
print('Satır hizası doğrulandı')
print('train/test:', train.shape, test.shape, '| ID örnek:', test_ids[:2])

y = pd.to_numeric(train[TARGET], errors='coerce').values
print(train[TARGET].describe().round(3))
if 'application_year' in train.columns and 'application_year' in test.columns:
    print('Train yıl dağılımı:', train['application_year'].value_counts(normalize=True).sort_index().round(3).to_dict())
    print('Test yıl dağılımı :', test['application_year'].value_counts(normalize=True).sort_index().round(3).to_dict())


## 1) Feature-rich tabular + Turkish transformer text FE

TF-IDF yok. Mentor metni için Türkçe BERT/ELECTRA OOF skorları ve sentence-transformer embedding bileşenleri kullanılır.


In [ ]:
# Hücre 2 — feature factory + transformer text features + OOF target encoding + drift ağırlıkları
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA

BASE_CAT = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
TECH_COLS = ['coding_score', 'problem_solving_score', 'data_structures_score', 'sql_score',
             'machine_learning_score', 'backend_score', 'frontend_score', 'cloud_score', 'devops_score']
CORE_DEV_COLS = ['coding_score', 'problem_solving_score', 'data_structures_score']
DATA_AI_COLS = ['sql_score', 'machine_learning_score', 'problem_solving_score']
WEB_DEV_COLS = ['backend_score', 'frontend_score', 'coding_score']
INFRA_COLS = ['cloud_score', 'devops_score', 'backend_score']
SOFT_COLS = ['communication_score', 'teamwork_score', 'leadership_score', 'presentation_score']
INTERVIEW_COLS = ['technical_interview_score', 'hr_interview_score']
PORTFOLIO_COLS = ['project_quality_score', 'portfolio_score', 'github_repo_count', 'github_avg_stars',
                  'open_source_contribution_count']
EXPERIENCE_COLS = ['real_client_project_count', 'internship_count', 'internship_duration_months',
                   'freelance_project_count', 'hackathon_count', 'hackathon_awards']
CAREER_PREP_COLS = ['linkedin_profile_score', 'cv_quality_score', 'certification_count', 'bootcamp_count',
                    'applications_sent', 'interviews_attended']
ACADEMIC_COLS = ['cgpa', 'english_exam_score', 'attendance_rate']
MISSING_FLAG_COLS = ['english_exam_score', 'internship_duration_months', 'github_avg_stars',
                     'open_source_contribution_count', 'hr_interview_score', 'linkedin_profile_score',
                     'portfolio_score']

ROLE_SKILL_MAP = {
    'Backend Developer': ['backend_score', 'coding_score', 'data_structures_score', 'sql_score', 'problem_solving_score'],
    'Frontend Developer': ['frontend_score', 'coding_score', 'presentation_score', 'communication_score'],
    'Software Developer': ['coding_score', 'backend_score', 'frontend_score', 'problem_solving_score', 'data_structures_score'],
    'Data Scientist': ['machine_learning_score', 'sql_score', 'problem_solving_score', 'data_structures_score'],
    'Data Analyst': ['sql_score', 'problem_solving_score', 'presentation_score', 'communication_score'],
    'AI Engineer': ['machine_learning_score', 'coding_score', 'problem_solving_score', 'data_structures_score'],
    'MLOps Engineer': ['machine_learning_score', 'devops_score', 'cloud_score', 'backend_score'],
    'DevOps Engineer': ['devops_score', 'cloud_score', 'backend_score', 'problem_solving_score'],
    'Cloud Engineer': ['cloud_score', 'devops_score', 'backend_score', 'problem_solving_score'],
    'Cybersecurity Analyst': ['devops_score', 'cloud_score', 'coding_score', 'problem_solving_score'],
    'Product Analyst': ['communication_score', 'presentation_score', 'sql_score', 'problem_solving_score'],
}

ROLE_SKILL_BROAD_MAP = {
    'Backend Developer': ['backend_score', 'coding_score', 'data_structures_score', 'sql_score', 'problem_solving_score', 'cloud_score', 'devops_score'],
    'Frontend Developer': ['frontend_score', 'coding_score', 'presentation_score', 'communication_score', 'portfolio_score'],
    'Software Developer': ['coding_score', 'backend_score', 'frontend_score', 'problem_solving_score', 'data_structures_score'],
    'Data Scientist': ['machine_learning_score', 'sql_score', 'problem_solving_score', 'data_structures_score', 'coding_score'],
    'Data Analyst': ['sql_score', 'problem_solving_score', 'presentation_score', 'communication_score', 'portfolio_score'],
    'AI Engineer': ['machine_learning_score', 'coding_score', 'problem_solving_score', 'data_structures_score'],
    'MLOps Engineer': ['machine_learning_score', 'devops_score', 'cloud_score', 'backend_score', 'coding_score', 'problem_solving_score'],
    'DevOps Engineer': ['devops_score', 'cloud_score', 'backend_score', 'problem_solving_score'],
    'Cloud Engineer': ['cloud_score', 'devops_score', 'backend_score', 'problem_solving_score', 'coding_score'],
    'Cybersecurity Analyst': ['devops_score', 'cloud_score', 'coding_score', 'problem_solving_score', 'sql_score'],
    'Product Analyst': ['communication_score', 'presentation_score', 'sql_score', 'problem_solving_score'],
}

POS_KEYWORDS = ['mükemmel', 'olağanüstü', 'başarı', 'başarılar', 'güçlü', 'yüksek', 'potansiyel',
                'etkileyici', 'yaratıcı', 'analitik', 'dikkat çekici', 'avantaj', 'katkı', 'liderlik',
                'kanıtlıyor', 'sergiliyor', 'öne çıkıyor']
NEG_KEYWORDS = ['eksik', 'eksikliği', 'geliştirmesi', 'geliştirmeli', 'çalışması', 'çalışmalı',
                'ihtiyaç', 'zorlaştırıyor', 'zorluk', 'düşük', 'başlangıç', 'risk', 'gerekiyor',
                'yetersiz', 'güçlendirmesi']


# Data-driven mentor template phrases mined from train text. These are not TF-IDF features;
# they are compact phrase-score features that capture mentor wording missed by the coarse keyword list.
MINED_POS_PHRASE_WEIGHTS = {
    'yüksek skoru': 15.5, 'mükemmel': 15.2, 'yüksek kalite': 15.0, 'aktif katkıları': 14.0,
    'sektörde aranan': 13.6, 'onu sektördeki': 13.5, 'yüksek proje': 13.2, 'aranan': 13.0,
    'olağanüstü': 12.7, 'kaliteli': 12.1, 'son derece': 11.9, 'gözler önüne seriyor': 11.5,
    'açık kaynak': 11.3, 'adaylardan': 11.2, 'başarılar elde': 11.2,
    'performans sergileyen': 11.0, 'github daki aktif': 10.8, 'github üzerindeki aktif': 11.6,
    'güçlü aday yapıyor': 10.6, 'sektördeki': 10.5, 'büyük avantaj sağlayacak': 10.3,
    'gelecekteki kariyerinde': 10.2, 'ayırıyor': 10.2, 'oldukça başarılı': 9.5,
    'potansiyelini ortaya koyuyor': 9.2, 'temel oluşturduğunu': 8.8,
}
MINED_NEG_PHRASE_WEIGHTS = {
    'belirli': 13.5, 'eksiklikler': 12.5, 'aşamasında': 12.0, 'becerilerinizde': 11.4,
    'göstermesi gerekiyor': 11.4, 'teknik becerilerde': 11.3, 'yapman gerektiğini': 11.3,
    'becerilerde fazla': 11.2, 'henüz': 10.6, 'var özellikle': 10.3, 'fazla projeye': 10.3,
    'gerekiyor': 9.9, 'kalitesinin artırılması': 9.6, 'becerilerde': 9.0, 'güzel': 9.0,
    'gerektiği aşikar': 8.7, 'sınırlı': 8.6, 'zorluklar': 8.6, 'gerektiği açık': 8.5,
    'deneyiminin': 8.4, 'ihtiyaç var': 8.3, 'engel teşkil': 8.2, 'teknik yeterliliklerini': 8.0,
    'deneyime ihtiyaç': 8.0, 'gelişim alanları': 7.9, 'gerekecek': 8.0, 'fazla deneyime ihtiyaç': 7.9,
    'üzerinde çalışması': 7.5, 'geliştirmesi': 7.0, 'çalışması gerekiyor': 8.5,
}



def existing(cols, df):
    return [c for c in cols if c in df.columns]


def numericize(df):
    skip = set(BASE_CAT + [ID_COL, 'mentor_feedback_text'])
    for c in df.columns:
        if c not in skip and c != TARGET:
            df[c] = pd.to_numeric(df[c], errors='ignore')
    return df


def add_mean_std(df, name, cols):
    cols = existing(cols, df)
    if not cols:
        return
    block = df[cols].apply(pd.to_numeric, errors='coerce')
    df[f'{name}_mean'] = block.mean(axis=1)
    df[f'{name}_std'] = block.std(axis=1).fillna(0)
    df[f'{name}_min'] = block.min(axis=1)
    df[f'{name}_max'] = block.max(axis=1)
    df[f'{name}_range'] = df[f'{name}_max'] - df[f'{name}_min']


def safe_div(a, b):
    return a / b.replace(0, np.nan)


def safe_hmean(df, cols):
    cols = existing(cols, df)
    if not cols:
        return pd.Series(np.nan, index=df.index)
    block = df[cols].apply(pd.to_numeric, errors='coerce').replace(0, np.nan)
    denom = (1.0 / block).sum(axis=1, min_count=len(cols))
    return len(cols) / denom.replace(0, np.nan)


def add_interaction(df, a, b, name=None, scale=100.0):
    if a in df.columns and b in df.columns:
        name = name or f'{a}_x_{b}'
        df[name] = pd.to_numeric(df[a], errors='coerce') * pd.to_numeric(df[b], errors='coerce') / scale


def add_role_fit(df):
    if 'target_role' not in df.columns:
        return
    tech_cols = existing(TECH_COLS, df)
    df['role_skill_fit'] = df[tech_cols].mean(axis=1) if tech_cols else np.nan
    for role, cols in ROLE_SKILL_MAP.items():
        cols = existing(cols, df)
        if cols:
            mask = df['target_role'].astype(str).eq(role)
            df.loc[mask, 'role_skill_fit'] = df.loc[mask, cols].mean(axis=1)
            df.loc[mask, 'role_skill_min'] = df.loc[mask, cols].min(axis=1)
            df.loc[mask, 'role_skill_max'] = df.loc[mask, cols].max(axis=1)
            df.loc[mask, 'role_skill_std'] = df.loc[mask, cols].std(axis=1).fillna(0)
    if 'tech_mean' in df.columns:
        df['role_skill_gap_vs_tech'] = df['role_skill_fit'] - df['tech_mean']
    if 'project_quality_score' in df.columns:
        df['role_project_fit'] = df['role_skill_fit'] * df['project_quality_score'] / 100.0
    if 'technical_interview_score' in df.columns:
        df['role_interview_fit'] = df['role_skill_fit'] * df['technical_interview_score'] / 100.0

    df['role_skill_broad_fit'] = df['role_skill_fit']
    for role, cols in ROLE_SKILL_BROAD_MAP.items():
        cols = existing(cols, df)
        if cols:
            mask = df['target_role'].astype(str).eq(role)
            df.loc[mask, 'role_skill_broad_fit'] = df.loc[mask, cols].mean(axis=1)
    if 'project_quality_score' in df.columns:
        df['role_broad_project_fit'] = df['role_skill_broad_fit'] * df['project_quality_score'] / 100.0
    if 'technical_interview_score' in df.columns:
        df['role_broad_interview_fit'] = df['role_skill_broad_fit'] * df['technical_interview_score'] / 100.0


def add_text_features(df):
    if 'mentor_feedback_text' not in df.columns:
        return
    txt = df['mentor_feedback_text'].fillna('').astype(str)
    low = txt.str.lower()
    tokens = low.str.findall(r'[a-zçğıöşü]+')
    df['mentor_char_count'] = txt.str.len()
    df['mentor_word_count'] = tokens.str.len()
    df['mentor_unique_word_count'] = tokens.apply(lambda xs: len(set(xs)))
    df['mentor_sentence_count'] = low.str.count(r'[.!?]+').clip(lower=1)
    df['mentor_avg_word_len'] = df['mentor_char_count'] / df['mentor_word_count'].replace(0, np.nan)
    df['mentor_lexical_diversity'] = df['mentor_unique_word_count'] / df['mentor_word_count'].replace(0, np.nan)
    df['mentor_pos_kw_count'] = sum(low.str.contains(k, regex=False).astype(int) for k in POS_KEYWORDS)
    df['mentor_neg_kw_count'] = sum(low.str.contains(k, regex=False).astype(int) for k in NEG_KEYWORDS)
    df['mentor_sentiment_balance'] = df['mentor_pos_kw_count'] - df['mentor_neg_kw_count']
    df['mentor_sentiment_ratio'] = (df['mentor_pos_kw_count'] + 1) / (df['mentor_neg_kw_count'] + 1)
    df['mentor_has_but'] = low.str.contains('ancak|fakat|ama', regex=True).astype(int)
    df['mentor_has_need'] = low.str.contains('ihtiyaç|gerek|çalışmalı|geliştirm', regex=True).astype(int)
    df['mentor_has_superlative'] = low.str.contains('mükemmel|olağanüstü|çok yüksek|çok güçlü', regex=True).astype(int)
    df['mentor_has_deficit'] = low.str.contains('eksik|düşük|zorluk|yetersiz', regex=True).astype(int)

    mined_pos_score = np.zeros(len(df), dtype=float)
    mined_neg_score = np.zeros(len(df), dtype=float)
    mined_pos_count = np.zeros(len(df), dtype=float)
    mined_neg_count = np.zeros(len(df), dtype=float)
    for phrase, weight in MINED_POS_PHRASE_WEIGHTS.items():
        hit = low.str.contains(phrase, regex=False).astype(float).values
        mined_pos_score += hit * weight
        mined_pos_count += hit
    for phrase, weight in MINED_NEG_PHRASE_WEIGHTS.items():
        hit = low.str.contains(phrase, regex=False).astype(float).values
        mined_neg_score += hit * weight
        mined_neg_count += hit
    df['mentor_mined_pos_phrase_count'] = mined_pos_count
    df['mentor_mined_neg_phrase_count'] = mined_neg_count
    df['mentor_mined_phrase_score'] = mined_pos_score - mined_neg_score
    df['mentor_mined_phrase_abs'] = mined_pos_score + mined_neg_score
    df['mentor_mined_phrase_ratio'] = (mined_pos_score + 1.0) / (mined_neg_score + 1.0)
    mined_phrase_patterns = {
        'mentor_phrase_elite': 'mükemmel|olağanüstü|sektörde aranan|son derece|yüksek kalite',
        'mentor_phrase_github_active': 'github daki aktif|github üzerindeki aktif|açık kaynak',
        'mentor_phrase_needs_work': 'gerekiyor|ihtiyaç|geliştirmesi|çalışması|eksiklik',
        'mentor_phrase_early_stage': 'henüz|aşamasında|sınırlı|belirli',
    }
    for col, pat in mined_phrase_patterns.items():
        df[col] = low.str.contains(pat, regex=True).astype(int)


    topic_patterns = {
        'mentor_topic_backend': 'backend|veritabanı|api',
        'mentor_topic_frontend': 'frontend|arayüz',
        'mentor_topic_data': 'veri bilimi|veri analizi|analitik|sql',
        'mentor_topic_ai_ml': 'ai|yapay zeka|makine öğrenimi|ml',
        'mentor_topic_devops_cloud': 'devops|cloud|bulut',
        'mentor_topic_security': 'siber güvenlik|security',
        'mentor_topic_project': 'proje|portföy|gerçek müşteri',
        'mentor_topic_soft': 'iletişim|takım|liderlik|sunum',
        'mentor_topic_experience': 'staj|hackathon|açık kaynak|freelance',
    }
    for col, pat in topic_patterns.items():
        df[col] = low.str.contains(pat, regex=True).astype(int)

    role_patterns = {
        'Backend Developer': ['backend', 'veritabanı', 'api'],
        'Frontend Developer': ['frontend', 'arayüz'],
        'Software Developer': ['yazılım', 'software'],
        'Data Scientist': ['veri bilimi', 'makine öğrenimi', 'analitik'],
        'Data Analyst': ['veri analizi', 'analitik', 'sql'],
        'AI Engineer': ['ai', 'yapay zeka', 'makine öğrenimi'],
        'MLOps Engineer': ['mlops', 'devops', 'cloud'],
        'DevOps Engineer': ['devops', 'cloud'],
        'Cloud Engineer': ['cloud', 'bulut'],
        'Cybersecurity Analyst': ['siber güvenlik', 'security'],
        'Product Analyst': ['ürün', 'product', 'analitik'],
    }
    if 'target_role' in df.columns:
        df['mentor_mentions_target_role'] = [
            int(any(p in text for p in role_patterns.get(role, [])))
            for role, text in zip(df['target_role'].astype(str), low)
        ]


def add_feature_rich(df):
    df = df.copy()
    df = numericize(df)

    for c in BASE_CAT:
        if c in df.columns:
            df[c] = df[c].fillna('__MISSING__').astype(str)

    for c in MISSING_FLAG_COLS:
        if c in df.columns:
            df[f'{c}_missing'] = df[c].isna().astype(int)
    miss_cols = [f'{c}_missing' for c in MISSING_FLAG_COLS if f'{c}_missing' in df.columns]
    if miss_cols:
        df['total_key_missing_count'] = df[miss_cols].sum(axis=1)

    if 'university_tier' in df.columns:
        tier_num = df['university_tier'].str.extract(r'(\d+)')[0].astype(float)
        df['university_tier_num'] = tier_num
        df['university_tier_strength'] = 5 - tier_num
    if 'application_year' in df.columns:
        df['application_year_idx'] = pd.to_numeric(df['application_year'], errors='coerce') - 2019
    if {'application_year', 'graduation_year'}.issubset(df.columns):
        df['years_since_graduation'] = pd.to_numeric(df['application_year'], errors='coerce') - pd.to_numeric(df['graduation_year'], errors='coerce')
        df['is_pre_graduation'] = (df['years_since_graduation'] < 0).astype(int)
        df['is_graduation_year'] = (df['years_since_graduation'] == 0).astype(int)
    if {'age', 'graduation_year', 'application_year'}.issubset(df.columns):
        df['age_at_graduation'] = pd.to_numeric(df['age'], errors='coerce') + pd.to_numeric(df['graduation_year'], errors='coerce') - pd.to_numeric(df['application_year'], errors='coerce')
    if 'age' in df.columns:
        df['age_bucket'] = pd.cut(pd.to_numeric(df['age'], errors='coerce'), bins=[0, 21, 23, 25, 99], labels=['<=21', '22-23', '24-25', '26+']).astype(str)

    add_mean_std(df, 'academic', ACADEMIC_COLS)
    add_mean_std(df, 'tech', TECH_COLS)
    add_mean_std(df, 'core_dev', CORE_DEV_COLS)
    add_mean_std(df, 'data_ai', DATA_AI_COLS)
    add_mean_std(df, 'web_dev', WEB_DEV_COLS)
    add_mean_std(df, 'infra', INFRA_COLS)
    add_mean_std(df, 'soft', SOFT_COLS)
    add_mean_std(df, 'interview', INTERVIEW_COLS)
    add_mean_std(df, 'portfolio', PORTFOLIO_COLS)
    add_mean_std(df, 'experience', EXPERIENCE_COLS)
    add_mean_std(df, 'career_prep', CAREER_PREP_COLS)

    add_role_fit(df)

    if {'tech_mean', 'soft_mean'}.issubset(df.columns):
        df['tech_soft_gap'] = df['tech_mean'] - df['soft_mean']
        df['tech_soft_product'] = df['tech_mean'] * df['soft_mean'] / 100.0
    if {'technical_interview_score', 'hr_interview_score'}.issubset(df.columns):
        df['interview_gap_technical_hr'] = df['technical_interview_score'] - df['hr_interview_score']
    if {'project_quality_score', 'portfolio_score'}.issubset(df.columns):
        df['portfolio_project_gap'] = df['portfolio_score'] - df['project_quality_score']
    if {'cgpa', 'failed_courses_count'}.issubset(df.columns):
        df['academic_risk_score'] = (4.0 - pd.to_numeric(df['cgpa'], errors='coerce')) * (1 + pd.to_numeric(df['failed_courses_count'], errors='coerce'))
    if {'attendance_rate', 'failed_courses_count'}.issubset(df.columns):
        df['attendance_failure_pressure'] = (100 - pd.to_numeric(df['attendance_rate'], errors='coerce')) * (1 + pd.to_numeric(df['failed_courses_count'], errors='coerce'))
    if {'applications_sent', 'interviews_attended'}.issubset(df.columns):
        df['interview_conversion_rate'] = safe_div(pd.to_numeric(df['interviews_attended'], errors='coerce'), pd.to_numeric(df['applications_sent'], errors='coerce'))
        df['applications_without_interview'] = pd.to_numeric(df['applications_sent'], errors='coerce') - pd.to_numeric(df['interviews_attended'], errors='coerce')
    if {'hackathon_awards', 'hackathon_count'}.issubset(df.columns):
        df['hackathon_award_rate'] = safe_div(pd.to_numeric(df['hackathon_awards'], errors='coerce'), pd.to_numeric(df['hackathon_count'], errors='coerce'))
    if {'internship_duration_months', 'internship_count'}.issubset(df.columns):
        df['months_per_internship'] = safe_div(pd.to_numeric(df['internship_duration_months'], errors='coerce'), pd.to_numeric(df['internship_count'], errors='coerce'))
        df['has_internship'] = (pd.to_numeric(df['internship_count'], errors='coerce') > 0).astype(int)
    if {'github_avg_stars', 'github_repo_count'}.issubset(df.columns):
        df['github_total_stars_proxy'] = pd.to_numeric(df['github_avg_stars'], errors='coerce') * pd.to_numeric(df['github_repo_count'], errors='coerce')
    if {'open_source_contribution_count', 'github_repo_count'}.issubset(df.columns):
        df['oss_per_repo'] = safe_div(pd.to_numeric(df['open_source_contribution_count'], errors='coerce'), pd.to_numeric(df['github_repo_count'], errors='coerce'))
    if {'real_client_project_count', 'freelance_project_count', 'hackathon_count'}.issubset(df.columns):
        df['external_project_count'] = pd.to_numeric(df['real_client_project_count'], errors='coerce') + pd.to_numeric(df['freelance_project_count'], errors='coerce') + pd.to_numeric(df['hackathon_count'], errors='coerce')
    if {'certification_count', 'bootcamp_count'}.issubset(df.columns):
        df['formal_learning_count'] = pd.to_numeric(df['certification_count'], errors='coerce') + pd.to_numeric(df['bootcamp_count'], errors='coerce')

    for a, b, name in [
        ('project_quality_score', 'technical_interview_score', 'project_x_technical_interview'),
        ('project_quality_score', 'portfolio_score', 'project_x_portfolio'),
        ('communication_score', 'technical_interview_score', 'communication_x_technical_interview'),
        ('communication_score', 'hr_interview_score', 'communication_x_hr_interview'),
        ('tech_mean', 'project_quality_score', 'tech_x_project_quality'),
        ('tech_mean', 'technical_interview_score', 'tech_x_technical_interview'),
        ('soft_mean', 'hr_interview_score', 'soft_x_hr_interview'),
        ('portfolio_mean', 'career_prep_mean', 'portfolio_x_career_prep'),
        ('experience_mean', 'project_quality_score', 'experience_x_project_quality'),
        ('role_skill_fit', 'technical_interview_score', 'role_fit_x_technical_interview'),
    ]:
        add_interaction(df, a, b, name=name)

    if {'project_quality_score', 'technical_interview_score'}.issubset(df.columns):
        df['project_interview_hmean'] = safe_hmean(df, ['project_quality_score', 'technical_interview_score'])
        df['project_interview_min'] = df[['project_quality_score', 'technical_interview_score']].apply(pd.to_numeric, errors='coerce').min(axis=1)
    if {'role_skill_fit', 'project_quality_score', 'technical_interview_score'}.issubset(df.columns):
        df['role_project_interview_hmean'] = safe_hmean(df, ['role_skill_fit', 'project_quality_score', 'technical_interview_score'])
        df['role_project_interview_min'] = df[['role_skill_fit', 'project_quality_score', 'technical_interview_score']].apply(pd.to_numeric, errors='coerce').min(axis=1)
    if {'role_skill_broad_fit', 'project_quality_score', 'technical_interview_score'}.issubset(df.columns):
        df['role_broad_project_interview_hmean'] = safe_hmean(df, ['role_skill_broad_fit', 'project_quality_score', 'technical_interview_score'])
    if {'real_client_project_count', 'project_quality_score'}.issubset(df.columns):
        df['client_project_quality'] = np.log1p(pd.to_numeric(df['real_client_project_count'], errors='coerce')) * df['project_quality_score']


    # Threshold/nonlinear tail features: mini-CV'de LGB wmse 82.2423 -> 82.1857.
    threshold_groups = {
        'tech': TECH_COLS,
        'soft': SOFT_COLS,
        'portfolio': PORTFOLIO_COLS,
        'experience': EXPERIENCE_COLS,
        'prep': CAREER_PREP_COLS,
        'academic': ACADEMIC_COLS,
    }
    for name, cols in threshold_groups.items():
        cols = existing(cols, df)
        if not cols:
            continue
        block = df[cols].apply(pd.to_numeric, errors='coerce')
        for thr in [50, 60, 70, 80, 90]:
            df[f'{name}_count_ge{thr}'] = (block >= thr).sum(axis=1)
        df[f'{name}_count_missing'] = block.isna().sum(axis=1)

    nonlinear_cols = existing([
        'project_quality_score', 'technical_interview_score', 'portfolio_score', 'role_skill_fit',
        'role_project_interview_hmean', 'project_interview_hmean', 'client_project_quality',
        'github_total_stars_proxy'
    ], df)
    for c in nonlinear_cols:
        x = pd.to_numeric(df[c], errors='coerce')
        df[f'{c}_sq'] = x * x / 100.0
        df[f'{c}_sqrt'] = np.sqrt(np.clip(x, 0, None))
        for thr in [70, 80, 90, 95]:
            df[f'{c}_ge{thr}'] = (x >= thr).astype(int)

    if {'project_quality_score', 'role_skill_fit', 'technical_interview_score', 'portfolio_score'}.issubset(df.columns):
        vals = df[['project_quality_score', 'role_skill_fit', 'technical_interview_score', 'portfolio_score']].apply(pd.to_numeric, errors='coerce')
        df['critical_top4_min'] = vals.min(axis=1)
        df['critical_top4_hmean'] = 4 / (1 / vals.replace(0, np.nan)).sum(axis=1, min_count=4)
        df['critical_top4_ge80_count'] = (vals >= 80).sum(axis=1)
        df['critical_top4_ge90_count'] = (vals >= 90).sum(axis=1)
        df['ceiling_proxy_top4'] = vals.prod(axis=1) / (100.0 ** 3)

    if {'project_quality_score', 'real_client_project_count', 'github_repo_count', 'open_source_contribution_count'}.issubset(df.columns):
        pq = pd.to_numeric(df['project_quality_score'], errors='coerce')
        real = np.log1p(pd.to_numeric(df['real_client_project_count'], errors='coerce'))
        gh = np.log1p(pd.to_numeric(df['github_repo_count'], errors='coerce'))
        oss = np.log1p(pd.to_numeric(df['open_source_contribution_count'], errors='coerce'))
        df['project_public_signal'] = pq * (1 + 0.35 * real + 0.15 * gh + 0.15 * oss)

    for c in ['applications_sent', 'interviews_attended', 'github_repo_count', 'github_avg_stars',
              'open_source_contribution_count', 'real_client_project_count', 'freelance_project_count',
              'hackathon_count', 'hackathon_awards', 'certification_count', 'bootcamp_count']:
        if c in df.columns:
            df[f'log1p_{c}'] = np.log1p(pd.to_numeric(df[c], errors='coerce'))

    if {'department', 'target_role'}.issubset(df.columns):
        df['department__target_role'] = df['department'].astype(str) + '__' + df['target_role'].astype(str)
    if {'university_tier', 'target_role'}.issubset(df.columns):
        df['tier__target_role'] = df['university_tier'].astype(str) + '__' + df['target_role'].astype(str)
    if {'age_bucket', 'target_role'}.issubset(df.columns):
        df['age_bucket__target_role'] = df['age_bucket'].astype(str) + '__' + df['target_role'].astype(str)
    if {'department', 'university_tier'}.issubset(df.columns):
        df['department__tier'] = df['department'].astype(str) + '__' + df['university_tier'].astype(str)

    add_text_features(df)
    if {'mentor_sentiment_balance', 'project_quality_score'}.issubset(df.columns):
        df['mentor_project_signal'] = pd.to_numeric(df['mentor_sentiment_balance'], errors='coerce') * pd.to_numeric(df['project_quality_score'], errors='coerce') / 100.0

    if {'mentor_mined_phrase_score', 'project_quality_score'}.issubset(df.columns):
        df['mentor_mined_score_x_project'] = pd.to_numeric(df['mentor_mined_phrase_score'], errors='coerce') * pd.to_numeric(df['project_quality_score'], errors='coerce') / 100.0
    if {'mentor_mined_phrase_score', 'role_skill_fit'}.issubset(df.columns):
        df['mentor_mined_score_x_rolefit'] = pd.to_numeric(df['mentor_mined_phrase_score'], errors='coerce') * pd.to_numeric(df['role_skill_fit'], errors='coerce') / 100.0
    return df


def add_frequency_encoding(train_df, test_df, cols):
    all_df = pd.concat([train_df[cols], test_df[cols]], axis=0, ignore_index=True)
    for c in cols:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        freq = all_df[c].astype(str).value_counts(normalize=True)
        train_df[f'{c}_freq'] = train_df[c].astype(str).map(freq).astype(float)
        test_df[f'{c}_freq'] = test_df[c].astype(str).map(freq).astype(float)
    return train_df, test_df


def add_group_rank_features(train_df, test_df):
    score_cols = existing([
        'project_quality_score', 'technical_interview_score', 'role_skill_fit', 'tech_mean',
        'portfolio_mean', 'experience_mean', 'career_prep_mean', 'mentor_sentiment_balance',
        'github_total_stars_proxy', 'external_project_count'
    ], train_df)
    group_cols = existing(['application_year', 'target_role', 'department__target_role', 'tier__target_role'], train_df)
    if not score_cols or not group_cols:
        return train_df, test_df
    all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    n_train = len(train_df)
    for g in group_cols:
        gname = re.sub(r'[^a-zA-Z0-9]+', '_', g).strip('_')
        for c in score_cols:
            cname = re.sub(r'[^a-zA-Z0-9]+', '_', c).strip('_')
            vals = pd.to_numeric(all_df[c], errors='coerce')
            grp = all_df[g].astype(str)
            rank_col = f'{cname}_rank_in_{gname}'
            z_col = f'{cname}_z_in_{gname}'
            all_df[rank_col] = vals.groupby(grp).rank(pct=True)
            mean = vals.groupby(grp).transform('mean')
            std = vals.groupby(grp).transform('std').replace(0, np.nan)
            all_df[z_col] = (vals - mean) / std
            train_df[rank_col] = all_df.loc[:n_train-1, rank_col].values
            test_df[rank_col] = all_df.loc[n_train:, rank_col].values
            train_df[z_col] = all_df.loc[:n_train-1, z_col].values
            test_df[z_col] = all_df.loc[n_train:, z_col].values
    return train_df, test_df


def cache_slug(text):
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')[:80]


def add_sentence_embedding_features(train_df, test_df):
    if not RUN_SENTENCE_EMBEDDINGS or 'mentor_feedback_text' not in train_df.columns:
        return train_df, test_df
    cache_name = f'sentence_embeddings_{cache_slug(SENTENCE_EMBEDDING_MODEL)}_{SENTENCE_EMBEDDING_COMPONENTS}.npz'
    cache_path = CACHE_DIR / cache_name
    read_cache_path = find_cache_file(cache_name)
    if read_cache_path is not None:
        data = np.load(read_cache_path, allow_pickle=True)
        comps_train, comps_test = data['train'], data['test']
        if read_cache_path != cache_path:
            np.savez_compressed(cache_path, train=comps_train, test=comps_test)
        print('Sentence embedding cache kullanıldı:', read_cache_path)
    else:
        try:
            from sentence_transformers import SentenceTransformer
            import torch
        except Exception as e:
            print('SentenceTransformer import edilemedi, embedding atlandı:', repr(e))
            return train_df, test_df
        texts = pd.concat([
            train_df['mentor_feedback_text'].fillna('').astype(str),
            test_df['mentor_feedback_text'].fillna('').astype(str)
        ], ignore_index=True).tolist()
        device = 'cuda' if HAS_CUDA else 'cpu'
        batch_size = 128 if device == 'cuda' else 32
        model = SentenceTransformer(SENTENCE_EMBEDDING_MODEL, device=device)
        emb = model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
        emb = np.asarray(emb, dtype=np.float32)
        n_comp = min(SENTENCE_EMBEDDING_COMPONENTS, emb.shape[1], len(emb) - 1)
        pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        comps = pca.fit_transform(emb).astype(np.float32)
        comps_train, comps_test = comps[:len(train_df)], comps[len(train_df):]
        np.savez_compressed(cache_path, train=comps_train, test=comps_test)
        print(f'Sentence embedding eklendi: {SENTENCE_EMBEDDING_MODEL} | dim={emb.shape[1]} | PCA={n_comp} | explained={pca.explained_variance_ratio_.sum():.3f}')
        del model, emb, comps
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
    for i in range(comps_train.shape[1]):
        col = f'st_emb_{i:02d}'
        train_df[col] = comps_train[:, i]
        test_df[col] = comps_test[:, i]
    train_df['st_emb_abs_mean'] = np.abs(comps_train).mean(axis=1)
    test_df['st_emb_abs_mean'] = np.abs(comps_test).mean(axis=1)
    train_df['st_emb_l2'] = np.sqrt((comps_train ** 2).sum(axis=1))
    test_df['st_emb_l2'] = np.sqrt((comps_test ** 2).sum(axis=1))
    return train_df, test_df


def add_transformer_oof_text_predictions(train_df, test_df, y_values, weights=None):
    if not RUN_TRANSFORMER_TEXT_OOF or 'mentor_feedback_text' not in train_df.columns:
        return train_df, test_df
    try:
        import torch
        from torch.utils.data import DataLoader, Dataset
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    except Exception as e:
        print('Transformers import edilemedi, OOF text modeli atlandı:', repr(e))
        return train_df, test_df

    device = 'cuda' if HAS_CUDA else 'cpu'
    if device != 'cuda' and not ALLOW_CPU_TRANSFORMER_OOF:
        msg = 'GPU/CUDA görünmüyor; transformer OOF fine-tune yapılmayacak. CUDA destekli PyTorch/driver kurulumunu düzeltmeden gece koşusu boşa gider.'
        if REQUIRE_CUDA_FOR_TEXT_OOF:
            raise RuntimeError(msg)
        print(msg + ' RUN_SENTENCE_EMBEDDINGS + keyword featureları devam ediyor.')
        return train_df, test_df

    class EncodedTextDataset(Dataset):
        def __init__(self, enc, labels=None, sample_weight=None):
            self.enc = enc
            self.labels = labels
            self.sample_weight = sample_weight
        def __len__(self):
            return len(self.enc['input_ids'])
        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
                sw = 1.0 if self.sample_weight is None else float(self.sample_weight[idx])
                item['sample_weight'] = torch.tensor(sw, dtype=torch.float32)
            return item

    def predict_batches(model, loader):
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch).logits.squeeze(-1)
                preds.append(out.detach().cpu().numpy())
        return np.concatenate(preds)

    texts_train = train_df['mentor_feedback_text'].fillna('').astype(str).tolist()
    texts_test = test_df['mentor_feedback_text'].fillna('').astype(str).tolist()
    y_scaled = np.asarray(y_values, dtype=np.float32) / 100.0
    if weights is None:
        sw_all = np.ones(len(y_scaled), dtype=np.float32)
    else:
        sw_all = np.asarray(weights, dtype=np.float32)
        sw_all = sw_all / np.nanmean(sw_all)
    kf_text = KFold(n_splits=TRANSFORMER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE + 19)
    use_amp = device == 'cuda'

    for model_id in TRANSFORMER_MODEL_IDS:
        slug = cache_slug(model_id)
        col = f'text_oof_{slug}'
        cache_name = f'{slug}_oof_{TRANSFORMER_N_SPLITS}fold_{TRANSFORMER_EPOCHS}ep_{TRANSFORMER_MAX_LENGTH}.npz'
        cache_path = CACHE_DIR / cache_name
        read_cache_path = find_cache_file(cache_name)
        if read_cache_path is not None:
            data = np.load(read_cache_path)
            train_df[col] = data['oof']
            test_df[col] = data['test']
            if read_cache_path != cache_path:
                np.savez_compressed(cache_path, oof=data['oof'], test=data['test'])
            print('Transformer OOF cache kullanıldı:', read_cache_path)
            continue

        print(f'Transformer OOF başlıyor: {model_id} | device={device}')
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        test_enc = tokenizer(texts_test, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
        test_loader = DataLoader(EncodedTextDataset(test_enc), batch_size=TRANSFORMER_BATCH_SIZE * 2, shuffle=False)
        oof = np.zeros(len(train_df), dtype=np.float32)
        test_pred = np.zeros(len(test_df), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(kf_text.split(texts_train), 1):
            tr_text = [texts_train[i] for i in tr_idx]
            va_text = [texts_train[i] for i in va_idx]
            tr_enc = tokenizer(tr_text, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
            va_enc = tokenizer(va_text, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
            tr_ds = EncodedTextDataset(tr_enc, y_scaled[tr_idx], sw_all[tr_idx])
            va_ds = EncodedTextDataset(va_enc)
            tr_loader = DataLoader(tr_ds, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True)
            va_loader = DataLoader(va_ds, batch_size=TRANSFORMER_BATCH_SIZE * 2, shuffle=False)

            model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=1, problem_type='regression')
            model.to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LR, weight_decay=TRANSFORMER_WEIGHT_DECAY)
            total_steps = max(1, TRANSFORMER_EPOCHS * len(tr_loader))
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps)
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

            for epoch in range(TRANSFORMER_EPOCHS):
                model.train()
                losses = []
                for batch in tr_loader:
                    labels = batch.pop('labels').to(device)
                    sample_weight = batch.pop('sample_weight').to(device)
                    batch = {k: v.to(device) for k, v in batch.items()}
                    optimizer.zero_grad(set_to_none=True)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        pred = model(**batch).logits.squeeze(-1)
                        loss = ((pred - labels) ** 2 * sample_weight).mean()
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    losses.append(float(loss.detach().cpu()))
                print(f'{slug} fold={fold} epoch={epoch+1}/{TRANSFORMER_EPOCHS} loss={np.mean(losses):.5f}')

            oof[va_idx] = np.clip(predict_batches(model, va_loader) * 100.0, 0, 100)
            test_pred += np.clip(predict_batches(model, test_loader) * 100.0, 0, 100) / TRANSFORMER_N_SPLITS
            del model, tr_enc, va_enc, tr_ds, va_ds, tr_loader, va_loader
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

        train_df[col] = oof
        test_df[col] = test_pred
        np.savez_compressed(cache_path, oof=oof, test=test_pred)
        print(f'{col}: MSE={np.mean((oof - y_values) ** 2):.4f} corr={np.corrcoef(oof, y_values)[0,1]:.4f}')
    return train_df, test_df


def weighted_group_mean(frame, key, y_values, weights=None):
    tmp = pd.DataFrame({key: frame[key].astype(str).values, '_y': y_values})
    if weights is None:
        return tmp.groupby(key)['_y'].mean()
    tmp['_w'] = weights
    tmp['_yw'] = tmp['_y'] * tmp['_w']
    sums = tmp.groupby(key)[['_yw', '_w']].sum()
    return sums['_yw'] / sums['_w']


def add_oof_target_encoding(train_df, test_df, cols, y_values, weights=None, n_splits=5):
    global_mean = np.average(y_values, weights=weights) if weights is not None else np.mean(y_values)
    kf_te = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE + 7)
    for c in cols:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        new_col = f'{c}_te'
        oof = np.full(len(train_df), global_mean, dtype=float)
        for tr_idx, va_idx in kf_te.split(train_df):
            means = weighted_group_mean(train_df.iloc[tr_idx], c, y_values[tr_idx], None if weights is None else weights[tr_idx])
            oof[va_idx] = train_df.iloc[va_idx][c].astype(str).map(means).fillna(global_mean).values
        full_means = weighted_group_mean(train_df, c, y_values, weights)
        train_df[new_col] = oof
        test_df[new_col] = test_df[c].astype(str).map(full_means).fillna(global_mean).values
    return train_df, test_df

before_cols = set(train.columns)
train = add_feature_rich(train)
test = add_feature_rich(test)
cat_for_unsup = [c for c in BASE_CAT + ['age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier'] if c in train.columns]
if RUN_FREQUENCY_ENCODING:
    train, test = add_frequency_encoding(train, test, cat_for_unsup)
else:
    print('Frequency encoding kapalı: *_freq kolonları noise-pruned modda üretilmiyor.')
train, test = add_group_rank_features(train, test)
train, test = add_sentence_embedding_features(train, test)

# Test dağılımını CV pusulasına yansıt: yıl + rol + tier kompozisyonunu shrink'li ağırlıklandır.
def _joined_group_key(df, groups):
    if len(groups) == 1:
        return df[groups[0]].astype(str)
    return df[groups].astype(str).agg('||'.join, axis=1)


def make_distribution_weights(train_df, test_df, groups, shrink=0.0):
    groups = [g for g in groups if g in train_df.columns and g in test_df.columns]
    if not groups:
        return np.ones(len(train_df), dtype=float), pd.DataFrame()
    tr = train_df.groupby(groups).size().rename('train_n')
    te = test_df.groupby(groups).size().rename('test_n')
    stats = pd.concat([tr, te], axis=1).fillna(0.0)
    stats['train_share'] = stats['train_n'] / len(train_df)
    stats['test_share'] = stats['test_n'] / len(test_df)
    stats['ratio_raw'] = (stats['test_share'] / stats['train_share'].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    stats['ratio'] = (stats['train_n'] * stats['ratio_raw'] + float(shrink)) / (stats['train_n'] + float(shrink)) if shrink else stats['ratio_raw']
    ratio_map = stats['ratio'].copy()
    ratio_map.index = ratio_map.index.map(lambda x: '||'.join(map(str, x)) if isinstance(x, tuple) else str(x))
    raw_w = _joined_group_key(train_df, groups).map(ratio_map).fillna(1.0).astype(float).values
    scale = np.mean(raw_w) if np.mean(raw_w) > 0 else 1.0
    stats['ratio_norm'] = stats['ratio'] / scale
    return raw_w / scale, stats.sort_values('ratio_norm', ascending=False)


if all(g in train.columns and g in test.columns for g in SAMPLE_WEIGHT_GROUPS):
    w, weight_stats = make_distribution_weights(train, test, SAMPLE_WEIGHT_GROUPS, shrink=SAMPLE_WEIGHT_SHRINK)
else:
    w, weight_stats = make_distribution_weights(train, test, ['application_year'], shrink=0.0)
print('Distribution weight groups:', SAMPLE_WEIGHT_GROUPS if len(weight_stats) else ['none'])
if len(weight_stats):
    print(weight_stats[['train_n', 'test_n', 'train_share', 'test_share', 'ratio_raw', 'ratio_norm']].head(12).round(3).to_string())
print(f'Sample weight summary: min={w.min():.3f} mean={w.mean():.3f} max={w.max():.3f} std={w.std():.3f}')

train, test = add_transformer_oof_text_predictions(train, test, y, weights=w)

te_cols = [c for c in BASE_CAT + ['age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier'] if c in train.columns]
train, test = add_oof_target_encoding(train, test, te_cols, y, weights=w, n_splits=5)

# Drift etkileşimleri: tekil güçlü sinyallerin zaman içinde değişen etkisini modele açık ver.
def add_drift(df):
    df = df.copy()
    if 'application_year' not in df.columns:
        return df
    yr = pd.to_numeric(df['application_year'], errors='coerce') - 2019
    base_cols = ['project_quality_score', 'tech_mean', 'technical_interview_score', 'bert_pred',
                 'role_skill_fit', 'portfolio_score', 'experience_mean', 'mentor_pos_kw_count',
                 'mentor_neg_kw_count', 'mentor_sentiment_balance', 'career_prep_mean',
                 'project_interview_hmean', 'role_project_interview_hmean', 'mentor_project_signal']
    base_cols += [c for c in df.columns if c.startswith('text_oof_')]
    for c in base_cols:
        if c in df.columns:
            df[f'{c}_x_year'] = pd.to_numeric(df[c], errors='coerce') * yr
    return df

train = add_drift(train)
test = add_drift(test)

if 'bert_pred' in train.columns and 'bert_pred' in test.columns:
    bp = pd.to_numeric(train['bert_pred'], errors='coerce')
    print(f'bert_pred: corr={bp.corr(train[TARGET]):.3f} | MSE={((bp-train[TARGET])**2).mean():.1f}'
          f' | train mean={bp.mean():.1f} test mean={pd.to_numeric(test["bert_pred"], errors="coerce").mean():.1f}')
else:
    print('bert_pred yok; mentor metni transformer OOF + sentence embedding + keyword featurelarıyla temsil ediliyor.')

new_cols = sorted(set(train.columns) - before_cols)
print(f'Yeni feature sayısı: {len(new_cols)}')
print('Örnek yeni featurelar:', new_cols[:40])


def wmse(y_true, y_pred):
    return float(np.average((np.asarray(y_true) - np.asarray(y_pred))**2, weights=w))


In [ ]:
# Hücre 3 — v33 niş feature enrichment (AutoGluon-only)
# Amaç: AutoGluon'a domain anlamı taşıyan, target-leak içermeyen, segment/drift farkını anlatan kolonlar vermek.

def _num(df, col, default=np.nan):
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    return pd.Series(default, index=df.index, dtype='float64')


def _safe_div_series(a, b):
    return a / b.replace(0, np.nan)


def _bucket(s, bins, labels):
    return pd.cut(pd.to_numeric(s, errors='coerce'), bins=bins, labels=labels, include_lowest=True).astype(str).replace('nan', '__MISSING__')


ROLE_FAMILY_MAP = {
    'Backend Developer': 'software_backend',
    'Frontend Developer': 'software_frontend',
    'Software Developer': 'software_fullstack',
    'Data Scientist': 'data_ai',
    'Data Analyst': 'data_analytics',
    'AI Engineer': 'data_ai',
    'MLOps Engineer': 'infra_ai',
    'DevOps Engineer': 'infra_cloud',
    'Cloud Engineer': 'infra_cloud',
    'Cybersecurity Analyst': 'security',
    'Product Analyst': 'product_data',
}
HIGH_VALUE_ROLES = {'Cloud Engineer', 'MLOps Engineer', 'DevOps Engineer', 'Backend Developer'}
LOW_MEAN_ROLES = {'Cybersecurity Analyst', 'Frontend Developer', 'Product Analyst'}


def add_v33_domain_features(df):
    df = df.copy()
    role = df['target_role'].astype(str) if 'target_role' in df.columns else pd.Series('__UNK__', index=df.index)
    app_year = _num(df, 'application_year')
    grad_year = _num(df, 'graduation_year')

    df['v33_is_recent_app_2024p'] = app_year.ge(2024).astype(int)
    df['v33_is_future_app_2025p'] = app_year.ge(2025).astype(int)
    df['v33_is_grad_2025p'] = grad_year.ge(2025).astype(int)
    df['v33_app_year_centered'] = app_year - 2022.5
    df['v33_grad_year_centered'] = grad_year - 2022.5
    df['v33_app_grad_gap'] = app_year - grad_year
    df['v33_age_at_grad_proxy'] = _num(df, 'age') + grad_year - app_year

    # Role-family categoricals: AutoGluon categorical splitlerde doğrudan kullanır.
    df['role_family'] = role.map(ROLE_FAMILY_MAP).fillna('other_role')
    df['role_is_high_value'] = role.isin(HIGH_VALUE_ROLES).astype(int)
    df['role_is_low_mean'] = role.isin(LOW_MEAN_ROLES).astype(int)
    if 'application_year' in df.columns:
        df['year__role_family'] = df['application_year'].astype(str) + '__' + df['role_family'].astype(str)

    tech_cols = existing(TECH_COLS, df)
    soft_cols = existing(SOFT_COLS, df)
    exp_cols = existing(EXPERIENCE_COLS, df)
    career_cols = existing(CAREER_PREP_COLS, df)
    portfolio_cols = existing(PORTFOLIO_COLS, df)

    tech_block = df[tech_cols].apply(pd.to_numeric, errors='coerce') if tech_cols else pd.DataFrame(index=df.index)
    soft_block = df[soft_cols].apply(pd.to_numeric, errors='coerce') if soft_cols else pd.DataFrame(index=df.index)
    exp_block = df[exp_cols].apply(pd.to_numeric, errors='coerce') if exp_cols else pd.DataFrame(index=df.index)
    career_block = df[career_cols].apply(pd.to_numeric, errors='coerce') if career_cols else pd.DataFrame(index=df.index)
    portfolio_block = df[portfolio_cols].apply(pd.to_numeric, errors='coerce') if portfolio_cols else pd.DataFrame(index=df.index)

    if tech_cols:
        df['v33_tech_top2_mean'] = tech_block.apply(lambda r: r.nlargest(min(2, r.notna().sum())).mean(), axis=1)
        df['v33_tech_bottom2_mean'] = tech_block.apply(lambda r: r.nsmallest(min(2, r.notna().sum())).mean(), axis=1)
        df['v33_tech_top_bottom_gap'] = df['v33_tech_top2_mean'] - df['v33_tech_bottom2_mean']
        df['v33_tech_ge80_count'] = tech_block.ge(80).sum(axis=1)
        df['v33_tech_ge90_count'] = tech_block.ge(90).sum(axis=1)
        df['v33_tech_lt50_count'] = tech_block.lt(50).sum(axis=1)
        df['v33_tech_floor50_share'] = tech_block.ge(50).mean(axis=1)

    if soft_cols:
        df['v33_soft_top2_mean'] = soft_block.apply(lambda r: r.nlargest(min(2, r.notna().sum())).mean(), axis=1)
        df['v33_soft_bottom2_mean'] = soft_block.apply(lambda r: r.nsmallest(min(2, r.notna().sum())).mean(), axis=1)
        df['v33_soft_consistency'] = 100 - soft_block.std(axis=1).fillna(0)

    # Role-specific skill fit: primary fit, broad fit, gap to industry-ready thresholds.
    role_primary = []
    role_broad = []
    role_min = []
    role_gap80 = []
    role_gap90 = []
    role_weak_count = []
    for idx, r in role.items():
        primary_cols = existing(ROLE_SKILL_MAP.get(r, []), df)
        broad_cols = existing(ROLE_SKILL_BROAD_MAP.get(r, primary_cols), df)
        pvals = pd.to_numeric(df.loc[idx, primary_cols], errors='coerce') if primary_cols else pd.Series(dtype=float)
        bvals = pd.to_numeric(df.loc[idx, broad_cols], errors='coerce') if broad_cols else pvals
        role_primary.append(float(pvals.mean()) if len(pvals) else np.nan)
        role_broad.append(float(bvals.mean()) if len(bvals) else np.nan)
        role_min.append(float(pvals.min()) if len(pvals) else np.nan)
        role_gap80.append(float(np.maximum(0, 80 - pvals).mean()) if len(pvals) else np.nan)
        role_gap90.append(float(np.maximum(0, 90 - pvals).mean()) if len(pvals) else np.nan)
        role_weak_count.append(int((pvals < 55).sum()) if len(pvals) else 0)
    df['v33_role_primary_fit'] = role_primary
    df['v33_role_broad_fit'] = role_broad
    df['v33_role_primary_min'] = role_min
    df['v33_role_gap_to_80'] = role_gap80
    df['v33_role_gap_to_90'] = role_gap90
    df['v33_role_weak_skill_count'] = role_weak_count
    if 'tech_mean' in df.columns:
        df['v33_role_fit_minus_tech_mean'] = df['v33_role_primary_fit'] - _num(df, 'tech_mean')

    # Project/portfolio/career readiness nonlinearities.
    project_q = _num(df, 'project_quality_score')
    portfolio = _num(df, 'portfolio_score')
    real_client = _num(df, 'real_client_project_count').fillna(0)
    freelance = _num(df, 'freelance_project_count').fillna(0)
    internship = _num(df, 'internship_count').fillna(0)
    intern_months = _num(df, 'internship_duration_months').fillna(0)
    hackathons = _num(df, 'hackathon_count').fillna(0)
    awards = _num(df, 'hackathon_awards').fillna(0)
    repos = _num(df, 'github_repo_count').fillna(0)
    stars = _num(df, 'github_avg_stars').fillna(0)
    oss = _num(df, 'open_source_contribution_count').fillna(0)

    df['v33_real_world_experience'] = real_client * 2.0 + freelance * 1.2 + internship * 1.5 + intern_months / 6.0
    df['v33_competitive_signal'] = hackathons + 2.0 * awards
    df['v33_github_quality_signal'] = np.log1p(repos) * np.log1p(stars) + 0.08 * oss
    df['v33_project_market_signal'] = project_q * np.log1p(df['v33_real_world_experience'])
    df['v33_portfolio_project_hmean'] = safe_hmean(df.assign(_project_q=project_q, _portfolio=portfolio), ['_project_q', '_portfolio'])
    df['v33_project_x_role_primary'] = project_q * df['v33_role_primary_fit'] / 100
    df['v33_portfolio_x_github'] = portfolio * df['v33_github_quality_signal'] / 10

    apps = _num(df, 'applications_sent').fillna(0)
    interviews = _num(df, 'interviews_attended').fillna(0)
    cvq = _num(df, 'cv_quality_score')
    linkedin = _num(df, 'linkedin_profile_score')
    df['v33_interview_rate'] = _safe_div_series(interviews, apps + 1)
    df['v33_applications_per_interview'] = _safe_div_series(apps + 1, interviews + 1)
    df['v33_career_visibility'] = (cvq.fillna(cvq.median()) + linkedin.fillna(linkedin.median())) / 2
    df['v33_career_visibility_x_interview_rate'] = df['v33_career_visibility'] * df['v33_interview_rate']
    df['v33_application_efficiency'] = interviews - 0.12 * apps

    tech_mean = _num(df, 'tech_mean') if 'tech_mean' in df.columns else tech_block.mean(axis=1)
    soft_mean = _num(df, 'soft_mean') if 'soft_mean' in df.columns else soft_block.mean(axis=1)
    interview_mean = df[existing(INTERVIEW_COLS, df)].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    df['v33_tech_soft_gap'] = tech_mean - soft_mean
    df['v33_tech_x_interview'] = tech_mean * interview_mean / 100
    df['v33_soft_x_hr'] = soft_mean * _num(df, 'hr_interview_score') / 100
    df['v33_role_fit_x_interview'] = df['v33_role_primary_fit'] * interview_mean / 100

    # Risk/readiness flags.
    df['v33_failed_courses_penalty'] = _num(df, 'failed_courses_count').fillna(0) * (100 - _num(df, 'attendance_rate').fillna(0)) / 100
    df['v33_academic_signal'] = _num(df, 'cgpa') * 25 + 0.20 * _num(df, 'english_exam_score') + 0.10 * _num(df, 'attendance_rate') - 3 * _num(df, 'failed_courses_count')
    miss_cols = existing(MISSING_FLAG_COLS, df)
    df['v33_key_missing_count'] = df[miss_cols].isna().sum(axis=1) if miss_cols else 0
    df['v33_sparse_profile_flag'] = ((repos <= 1).astype(int) + portfolio.isna().astype(int) + linkedin.isna().astype(int) + (real_client <= 0).astype(int))

    # Buckets as categorical interaction handles for AutoGluon.
    df['v33_project_bucket'] = _bucket(project_q, [-0.1, 40, 60, 75, 90, 100.1], ['p0_40', 'p40_60', 'p60_75', 'p75_90', 'p90_100'])
    df['v33_role_fit_bucket'] = _bucket(df['v33_role_primary_fit'], [-0.1, 45, 60, 75, 88, 100.1], ['r0_45', 'r45_60', 'r60_75', 'r75_88', 'r88_100'])
    df['v33_interview_bucket'] = _bucket(interview_mean, [-0.1, 40, 60, 75, 90, 100.1], ['i0_40', 'i40_60', 'i60_75', 'i75_90', 'i90_100'])
    if 'application_year' in df.columns:
        df['v33_year__rolefit_bucket'] = df['application_year'].astype(str) + '__' + df['v33_role_fit_bucket'].astype(str)
        df['v33_year__project_bucket'] = df['application_year'].astype(str) + '__' + df['v33_project_bucket'].astype(str)

    return df


train = add_v33_domain_features(train)
test = add_v33_domain_features(test)


def add_group_percentile_features(train_df, test_df, group_cols, value_cols, prefix, min_count=20):
    group_cols = [c for c in group_cols if c in train_df.columns and c in test_df.columns]
    value_cols = [c for c in value_cols if c in train_df.columns and c in test_df.columns]
    if not group_cols or not value_cols:
        return train_df, test_df
    both = pd.concat([
        train_df[group_cols + value_cols].assign(_is_train=1),
        test_df[group_cols + value_cols].assign(_is_train=0),
    ], axis=0, ignore_index=True)
    group_key = both[group_cols].astype(str).agg('||'.join, axis=1)
    for col in value_cols:
        vals = pd.to_numeric(both[col], errors='coerce')
        pct = vals.groupby(group_key).rank(pct=True)
        counts = vals.groupby(group_key).transform('count')
        global_pct = vals.rank(pct=True)
        pct = pct.where(counts >= min_count, global_pct)
        new_col = f'{prefix}_{col}_pct'
        train_df[new_col] = pct.iloc[:len(train_df)].values
        test_df[new_col] = pct.iloc[len(train_df):].values
    return train_df, test_df

rank_value_cols = [
    'project_quality_score', 'technical_interview_score', 'portfolio_score', 'v33_role_primary_fit',
    'v33_project_market_signal', 'v33_career_visibility', 'v33_tech_x_interview',
    'mentor_sentiment_balance'
] + [c for c in train.columns if c.startswith('text_oof_')]
if RUN_V33_GROUP_PERCENTILE_FEATURES:
    for group_cols, prefix in [
        (['target_role'], 'rank_role'),
        (['application_year'], 'rank_year'),
        (['university_tier'], 'rank_tier'),
        (['application_year', 'target_role'], 'rank_year_role'),
        (['department', 'target_role'], 'rank_dept_role'),
    ]:
        train, test = add_group_percentile_features(train, test, group_cols, rank_value_cols, prefix=prefix, min_count=25)
else:
    print('v33 ekstra group-percentile rank featureları kapalı; feature factory içindeki test-independent/domain ranklar kalır.')

v33_cols = sorted([c for c in train.columns if c.startswith('v33_') or c.startswith('rank_') or c in ['role_family', 'year__role_family']])
print(f'v33 niş feature sayısı: {len(v33_cols)}')
print('v33 örnek featurelar:', v33_cols[:50])


In [ ]:
# Hücre 4 — AutoGluon model matrisi ve feature metadata
from sklearn.metrics import mean_squared_error

CAT = [
    c for c in BASE_CAT + [
        'age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier',
        'role_family', 'year__role_family', 'v33_project_bucket', 'v33_role_fit_bucket', 'v33_interview_bucket',
        'v33_year__rolefit_bucket', 'v33_year__project_bucket'
    ] if c in train.columns and c in test.columns
]
DROP_FOR_MODEL = {TARGET, ID_COL, 'mentor_feedback_text'}
feats = [c for c in train.columns if c not in DROP_FOR_MODEL and c in test.columns]

# Public-shift temizliği: combo kategorikler local random CV'de cazip görünse de 2024+/2025+ holdout'ta en net gürültü oldu.
# Bu blok ham role/department/tier/year bilgisini korur, sadece __ ile üretilen kombinasyonları ve onların TE/rank türevlerini düşürür.
if DROP_CATEGORICAL_INTERACTION_FEATURES:
    combo_noise_cols = [c for c in feats if '__' in c]
    if combo_noise_cols:
        print(f'Combo interaction featureları çıkarıldı: {len(combo_noise_cols)} | örnek={combo_noise_cols[:20]}')
        feats = [c for c in feats if c not in combo_noise_cols]
        CAT = [c for c in CAT if c not in combo_noise_cols]

# Text tarafında transformer OOF + sentence embedding + sentiment/phrase featureları kalır.
# Sadece ham topic/role mention bayrakları scout'ta stabil gürültü verdiği için düşürülür.
if DROP_TOPIC_ROLE_TEXT_FLAGS:
    topic_noise_cols = [c for c in feats if c.startswith('mentor_topic') or c.startswith('mentor_mentions')]
    if topic_noise_cols:
        print(f'Topic/role text flagları çıkarıldı: {len(topic_noise_cols)} | örnek={topic_noise_cols[:20]}')
        feats = [c for c in feats if c not in topic_noise_cols]
        CAT = [c for c in CAT if c not in topic_noise_cols]

obj_not_cat = [c for c in feats if str(train[c].dtype) in ('object', 'string') and c not in CAT]
if obj_not_cat:
    print('Model dışı bırakılan işlenmemiş object kolonları:', obj_not_cat)
    feats = [c for c in feats if c not in obj_not_cat]

constant_cols = [c for c in feats if train[c].nunique(dropna=False) <= 1 and test[c].nunique(dropna=False) <= 1]
if constant_cols:
    print('Sabit featurelar çıkarıldı:', constant_cols[:30], '... toplam', len(constant_cols))
    feats = [c for c in feats if c not in constant_cols]

for c in CAT:
    train[c] = train[c].fillna('__MISSING__').astype(str)
    test[c] = test[c].fillna('__MISSING__').astype(str)

# Sonsuz değerleri NaN'a çevir; AutoGluon NaN'ı yönetir.
for df in (train, test):
    num_cols = [c for c in feats if c not in CAT]
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

cat_idx = [feats.index(c) for c in CAT if c in feats]
print(f'AutoGluon feature sayısı: {len(feats)} | kategorik: {len(cat_idx)}')
print('Kategorikler:', CAT)

# Segment ağırlıkları ve drift raporu: resmi seçim MSE, eğitim ise test kompozisyonunu görsün.
def mse(y_true, y_pred):
    return float(mean_squared_error(y_true, np.clip(y_pred, 0, 100)))

def wmse_metric(y_true, y_pred):
    return float(np.average((np.asarray(y_true) - np.clip(np.asarray(y_pred), 0, 100))**2, weights=w))

print('Train/test yıl dağılımı:')
print(pd.concat([
    train['application_year'].value_counts(normalize=True).rename('train'),
    test['application_year'].value_counts(normalize=True).rename('test')
], axis=1).sort_index().round(4).to_string())
print('Train/test target_role dağılımı:')
print(pd.concat([
    train['target_role'].value_counts(normalize=True).rename('train'),
    test['target_role'].value_counts(normalize=True).rename('test')
], axis=1).round(4).to_string())


## 2) AutoGluon-only training

Manuel model yok. AutoGluon 5-fold bagging ile çalışır. Son hak sürümünde `AG_NUM_STACK_LEVELS=0` tutulur; bu daha stabil ve daha az disk/time risklidir.


In [ ]:
# Hücre 5 — SADECE AutoGluon eğitimi + submission
from autogluon.tabular import TabularPredictor

train_ag = train[feats + [TARGET]].copy()
test_ag = test[feats].copy()
train_ag['sample_weight'] = w

for c in [c for c in CAT if c in train_ag.columns and c in test_ag.columns]:
    categories = pd.Index(pd.concat([train_ag[c], test_ag[c]], axis=0).astype(str).unique())
    dtype = pd.CategoricalDtype(categories=categories)
    train_ag[c] = train_ag[c].astype(str).astype(dtype)
    test_ag[c] = test_ag[c].astype(str).astype(dtype)

ag_num_gpus = int(AG_TABULAR_NUM_GPUS) if HAS_CUDA else 0
ag_path = f'AutogluonModels_{NOTEBOOK_VERSION}_local' if LOCAL_RUN else f'AutogluonModels_{NOTEBOOK_VERSION}'
print(f'AutoGluon-only num_gpus={ag_num_gpus} | path={ag_path} | time_limit={TIME_LIMIT}s')

predictor = TabularPredictor(
    label=TARGET,
    eval_metric='root_mean_squared_error',
    sample_weight='sample_weight',
    weight_evaluation=AG_WEIGHT_EVALUATION,
    path=ag_path,
).fit(
    train_data=train_ag,
    time_limit=TIME_LIMIT,
    presets=AG_PRESETS,
    num_bag_folds=AG_NUM_BAG_FOLDS,
    num_stack_levels=AG_NUM_STACK_LEVELS,
    dynamic_stacking=False,
    num_gpus=ag_num_gpus,
    excluded_model_types=AG_EXCLUDED_MODEL_TYPES,
    verbosity=2,
)

lb = predictor.leaderboard(silent=True)
lb.to_csv('autogluon_leaderboard.csv', index=False)
print('AutoGluon leaderboard top 20:')
print(lb[['model', 'score_val', 'pred_time_val', 'fit_time']].head(20).to_string(index=False))

pred_ag = np.clip(predictor.predict(test_ag).values, 0, 100)
try:
    oof_ag = np.clip(predictor.predict_oof().values, 0, 100)
except AttributeError:
    oof_ag = np.clip(predictor.get_oof_pred().values, 0, 100)

print(f'AutoGluon OOF MSE = {mse(y, oof_ag):.4f} | drift-weighted MSE = {wmse_metric(y, oof_ag):.4f}')

# Public yıl dağılımı 2024-2026 ağırlıklı olduğu için AutoGluon'un unweighted ensemble'ını
# top base modellerin OOF tahminleriyle yeniden ağırlıklandır. Model aileleri yine sadece AutoGluon'dan gelir.
def make_year_weights(train_df, test_df):
    if 'application_year' not in train_df.columns or 'application_year' not in test_df.columns:
        return np.ones(len(train_df), dtype=float)
    tr_share = train_df['application_year'].value_counts(normalize=True)
    te_share = test_df['application_year'].value_counts(normalize=True)
    ratio = (te_share / tr_share).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    yy = train_df['application_year'].map(ratio).fillna(1.0).astype(float).values
    return yy / np.mean(yy)

year_w = make_year_weights(train, test)
print(f'Year-weight summary: min={year_w.min():.3f} mean={year_w.mean():.3f} max={year_w.max():.3f} std={year_w.std():.3f}')

def get_oof_for_model(model_name):
    try:
        return np.clip(predictor.predict_oof(model=model_name).values, 0, 100)
    except Exception as e:
        print(f'OOF alınamadı, model atlandı: {model_name} | {repr(e)}')
        return None

def optimize_convex_weights(P, y_true, weights, seed=42, n_random=6000):
    rng = np.random.default_rng(seed)
    k = P.shape[1]
    candidates = []
    candidates.append(np.ones(k) / k)
    for i in range(k):
        e = np.zeros(k); e[i] = 1.0; candidates.append(e)
    # Dirichlet araması: düşük boyutta sağlam, scipy gerektirmez.
    for alpha in [0.2, 0.5, 1.0, 2.0, 5.0]:
        candidates.extend(rng.dirichlet(np.ones(k) * alpha, size=max(1, n_random // 5)))
    best_w, best_score = None, np.inf
    yv = np.asarray(y_true, dtype=float)
    ww = np.asarray(weights, dtype=float)
    for cand in candidates:
        pred = np.clip(P @ cand, 0, 100)
        score = float(np.average((yv - pred) ** 2, weights=ww))
        if score < best_score:
            best_score, best_w = score, cand
    return best_w, best_score

base_models = [m for m in lb['model'].tolist() if 'WeightedEnsemble' not in str(m)]
base_models = base_models[:14]
model_oof, model_test, model_names = [], [], []
for mname in base_models:
    oof_m = get_oof_for_model(mname)
    if oof_m is None or len(oof_m) != len(train):
        continue
    try:
        pred_m = np.clip(predictor.predict(test_ag, model=mname).values, 0, 100)
    except Exception as e:
        print(f'Test pred alınamadı, model atlandı: {mname} | {repr(e)}')
        continue
    model_names.append(mname)
    model_oof.append(oof_m)
    model_test.append(pred_m)

pred_year_ens = pred_ag.copy()
if len(model_names) >= 2:
    P = np.vstack(model_oof).T
    T = np.vstack(model_test).T
    opt_w_year, opt_score_year = optimize_convex_weights(P, y, year_w, seed=RANDOM_STATE + 301, n_random=8000)
    opt_w_mse, opt_score_mse = optimize_convex_weights(P, y, np.ones(len(y)), seed=RANDOM_STATE + 302, n_random=8000)
    pred_year_ens = np.clip(T @ opt_w_year, 0, 100)
    pred_mse_ens = np.clip(T @ opt_w_mse, 0, 100)
    oof_year_ens = np.clip(P @ opt_w_year, 0, 100)
    oof_mse_ens = np.clip(P @ opt_w_mse, 0, 100)
    ens_info = pd.DataFrame({'model': model_names, 'weight_year': opt_w_year, 'weight_mse': opt_w_mse})
    ens_info.to_csv('autogluon_reweighted_ensemble_weights.csv', index=False)
    print('\nReweighted AutoGluon ensemble weights:')
    print(ens_info.sort_values('weight_year', ascending=False).head(20).round(5).to_string(index=False))
    print(f'Reweighted year-OOF weighted MSE = {opt_score_year:.4f} | plain MSE = {mse(y, oof_year_ens):.4f}')
    print(f'Reweighted mse-OOF plain MSE    = {opt_score_mse:.4f} | year-weighted MSE = {float(np.average((y-oof_mse_ens)**2, weights=year_w)):.4f}')
else:
    print('Yeterli base model OOF alınamadı; reweighted ensemble atlandı.')

def segment_report(pred, col, topn=20):
    if col not in train.columns:
        return pd.DataFrame()
    tmp = pd.DataFrame({col: train[col].astype(str), 'y': y, 'pred': pred, 'w': w})
    tmp['err'] = (tmp['y'] - tmp['pred']) ** 2
    tmp['abs_err'] = np.abs(tmp['y'] - tmp['pred'])
    rows = []
    for val, g in tmp.groupby(col):
        rows.append({
            col: val,
            'n': len(g),
            'target_mean': np.average(g['y'], weights=g['w']),
            'pred_mean': np.average(g['pred'], weights=g['w']),
            'bias_y_minus_pred': np.average(g['y'] - g['pred'], weights=g['w']),
            'mse': np.average(g['err'], weights=g['w']),
            'abs_err': np.average(g['abs_err'], weights=g['w']),
        })
    rep = pd.DataFrame(rows).sort_values('mse', ascending=False).reset_index(drop=True)
    print(f'\nSegment diagnostics by {col}:')
    print(rep.head(topn).round(4).to_string(index=False))
    return rep

seg_year = segment_report(oof_ag, 'application_year')
seg_role = segment_report(oof_ag, 'target_role')
seg_tier = segment_report(oof_ag, 'university_tier')
seg_year.to_csv('segment_report_year.csv', index=False)
seg_role.to_csv('segment_report_role.csv', index=False)
seg_tier.to_csv('segment_report_tier.csv', index=False)

out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
submission = pd.DataFrame({ID_COL: test_ids, TARGET: pred_year_ens})
submission[TARGET] = submission[TARGET].clip(0, 100)
submission.to_csv(out / 'submission.csv', index=False)
submission.to_csv(out / 'submission_reweighted_year.csv', index=False)
pd.DataFrame({ID_COL: test_ids, TARGET: pred_ag}).to_csv(out / 'submission_autogluon_only.csv', index=False)
if 'pred_mse_ens' in globals():
    pd.DataFrame({ID_COL: test_ids, TARGET: pred_mse_ens}).to_csv(out / 'submission_reweighted_mse.csv', index=False)

# Ana submit: public yıl dağılımına göre yeniden ağırlıklandırılmış AutoGluon ensemble.
print('\nSubmission sample:')
print(submission.head(5))
print('Yazılan dosyalar:', out / 'submission.csv', '|', out / 'submission_reweighted_year.csv', '|', out / 'submission_autogluon_only.csv', '| autogluon_leaderboard.csv')
print(f'Toplam süre: {(time.time() - t0) / 60:.1f} dk')


## Çalıştırma

Kaggle’da GPU ve Internet açıkken Run All yap. Varsayılan `KAGGLE_TIME_LIMIT=21600` yani 6 saat AutoGluon bütçesidir. Ana submit dosyası `submission.csv`. Bu v39 sürümü v38 ensemble mantığını korur; ek olarak BERT/ELECTRA yanına `xlm-roberta-base` OOF text regresyonu ekler ve tüm `text_oof_*` kolonlarını yıl etkileşimleriyle modele verir. Güvenli karşılaştırma için `submission_autogluon_only.csv`, `submission_reweighted_year.csv` ve `submission_reweighted_mse.csv` yazılır.
